# 움직임 시험 2 — 고정 스타일 참조(IP-Adapter)·이전 장 참조로 장 사이 깜빡임 줄이기 (Kaggle T4, 학습 없음)
시험 1(장마다 따로 그림→그림)과 같은 블로킹 두 가지(num: 숫자 블로킹 15장, clip: 나카무라 실루엣 블로킹 12장)를 세 방식으로 그린다.
- **V1 따로**: 장마다 따로 그림→그림(시험 1과 같음, 비교 기준).
- **V2 고정 참조**: 정점 장을 먼저 V1로 그리고, 그 그림을 IP-Adapter(그림을 참고해 화풍·색을 맞추는 모듈)로 모든 장에 참조.
- **V3 이전 장 참조**: 바로 앞 장의 결과를 참조(사슬). 블로킹이 크게 바뀌는 장(임팩트·컷)에서는 참조를 고정 참조로 되돌림.
- 같은 LoRA(컬러 3차 3바퀴, diffusers로 합침 — 서버에서 788개 모듈 모두 적용 확인), 같은 글·시드(1234), 강도 0.6.
- 설정(Kaggle): **GPU T4 x2**, **Internet On** → **Save Version → Save & Run All (Commit)**. 약 40–60분.
- Colab에서도 그대로 돈다: 런타임 유형 **T4 GPU** → 모두 실행. 마지막 칸이 결과 zip을 내려받게 한다(Colab은 창을 닫으면 결과가 사라지니 받은 뒤 닫을 것).
- 끝나면 Output의 `motion2_result.zip`(나란히 영상 `motion2.mp4`, 장별 그림, 깜빡임 수치)을 Drive 공유 링크로 알려 준다.


In [ ]:
# 0) 어디서 돌리나: Kaggle이면 /kaggle/working, Colab이면 /content/out 에 결과를 쓴다
import os, glob
COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
OUT = "/content/out" if COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
os.makedirs(OUT, exist_ok=True); print("결과 폴더", OUT)
# 1) 패키지와 자료: 기본 모델(HF), LoRA 3차 3바퀴·블로킹 두 가지(Drive)
# 버전 고정: Colab·Kaggle 기본 파이썬 3.13 + 새 transformers에서 diffusers 0.31이 깨졌다(FLAX_WEIGHTS_NAME, 2026-10-26). 서버 py3.13에서 아래 조합 import 확인
!pip install -q "diffusers==0.35.1" "transformers==4.56.2" "peft==0.17.1" "accelerate>=1.0"
!curl -sSL -C - -o /tmp/ilxl.safetensors https://huggingface.co/OnomaAIResearch/Illustrious-XL-v1.0/resolve/main/Illustrious-XL-v1.0.safetensors
!curl -sSL -o /tmp/lora.safetensors "https://drive.usercontent.google.com/download?id=1ht_UqE8JkJTUX-eO5lNIqvhyFRcprUWJ&export=download&confirm=t"
import os, glob
for name, fid in [("num", "1lLCSZbiHE0QUO6xSw_uIeWiyUoNjlgDx"), ("clip", "1jITc3rXRduBnoW3C05I2XLmJ_7hEdtI6")]:
    os.system(f'curl -sSL -o /tmp/{name}.zip "https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t"')
    os.makedirs(f"/tmp/blk_{name}", exist_ok=True)
    os.system(f"cd /tmp/blk_{name} && unzip -q -o /tmp/{name}.zip '*.png'")
    print(name, len(glob.glob(f"/tmp/blk_{name}/*.png")), "장")
!ls -la /tmp/ilxl.safetensors /tmp/lora.safetensors


In [ ]:
# 2) 파이프라인: SDXL 그림→그림 + LoRA 합치기 + IP-Adapter(SDXL plus, ViT-H 그림 인코더)
import torch
from diffusers import StableDiffusionXLImg2ImgPipeline, EulerAncestralDiscreteScheduler
from PIL import Image
pipe = StableDiffusionXLImg2ImgPipeline.from_single_file("/tmp/ilxl.safetensors", torch_dtype=torch.float16)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.load_lora_weights("/tmp/lora.safetensors", adapter_name="fx")
pipe.fuse_lora(lora_scale=1.0); pipe.unload_lora_weights()
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="sdxl_models", weight_name="ip-adapter-plus_sdxl_vit-h.safetensors",
                     image_encoder_folder="models/image_encoder")
pipe.to("cuda")
pipe.vae.to(torch.float32)  # VAE는 fp32(검은 그림·NaN 방지, 시험 1의 --no_half_vae와 같음)
print(torch.cuda.memory_allocated() / 1e9, "GB")


In [ ]:
# 3) 세 방식으로 그리기
import numpy as np, json
P = "nkmr_fx, vivid colors, anime screencap, explosion, fire, embers, white core orange rim, layered cel shading, no outline, sharp tips, large effect, no humans, city, night, wide shot"
N = "lowres, worst quality, text, watermark"
S, STEPS, CFG, SEED, IPS = 0.6, 28, 6.0, 1234, 0.6
def run(init, ref, scale):
    pipe.set_ip_adapter_scale(scale)
    g = torch.Generator("cuda").manual_seed(SEED)
    with torch.inference_mode():
        return pipe(prompt=P, negative_prompt=N, image=init, strength=S, num_inference_steps=STEPS, guidance_scale=CFG,
                    ip_adapter_image=ref, generator=g, width=1344, height=768).images[0]
def diff(a, b):
    return float(np.abs(np.asarray(a, np.float32) - np.asarray(b, np.float32)).mean())
log = {}
for name in ("num", "clip"):
    blk = [Image.open(p).convert("RGB") for p in sorted(glob.glob(f"/tmp/blk_{name}/*.png"))]
    area = [float((np.asarray(b, np.float32).max(2) > 120).mean()) for b in blk]
    peak = int(np.argmax(area))
    out = {v: [] for v in ("V1", "V2", "V3")}
    cache = {}
    for i, b in enumerate(blk):  # V1: 따로(참조 세기 0, 그림은 자리만 채움). 같은 블로킹(걸이 장)은 같은 결과를 다시 씀
        key = np.asarray(b).tobytes()
        if key not in cache:
            cache[key] = run(b, b, 0.0)
        out["V1"].append(cache[key])
    ref = out["V1"][peak]
    cache = {}
    for b in blk:  # V2: 정점 장 결과를 고정 참조
        key = np.asarray(b).tobytes()
        if key not in cache:
            cache[key] = run(b, ref, IPS)
        out["V2"].append(cache[key])
    prev = ref
    for i, b in enumerate(blk):  # V3: 앞 장 결과 참조, 블로킹이 크게 바뀌면(평균 차 > 40) 고정 참조로 되돌림
        if i > 0 and np.array_equal(np.asarray(b), np.asarray(blk[i - 1])):
            out["V3"].append(out["V3"][-1]); continue
        r = ref if (i == 0 or diff(b, blk[i - 1]) > 40) else prev
        im = run(b, r, IPS); out["V3"].append(im); prev = im
    for v, ims in out.items():
        d = f"{OUT}/m2/{name}/{v}"; os.makedirs(d, exist_ok=True)
        for i, im in enumerate(ims):
            im.save(f"{d}/f{i:03d}.png")
        # 깜빡임: 원본(블로킹)에 없는 장간 변화 = |Δ결과 − Δ블로킹| 평균
        fl = np.mean([abs(diff(ims[i + 1], ims[i]) - diff(blk[i + 1], blk[i])) for i in range(len(ims) - 1)])
        log[f"{name} {v}"] = round(float(fl), 2)
        print(name, v, len(ims), "장, 깜빡임", log[f"{name} {v}"])
    blk_d = f"{OUT}/m2/{name}/blocking"; os.makedirs(blk_d, exist_ok=True)
    for i, b in enumerate(blk):
        b.save(f"{blk_d}/f{i:03d}.png")
json.dump(log, open(f"{OUT}/motion2_flicker.json", "w"), ensure_ascii=False, indent=1)


In [ ]:
# 4) 나란히 영상: 위 줄 num, 아래 줄 clip / 칸 = 블로킹 | V1 따로 | V2 고정 참조 | V3 이전 장 참조. 24fps, 3번 반복
import cv2
rows = {n: {k: sorted(glob.glob(f"{OUT}/m2/{n}/{k}/*.png")) for k in ("blocking", "V1", "V2", "V3")} for n in ("num", "clip")}
def rd(p, label):
    im = cv2.resize(cv2.imread(p), (336, 192)); cv2.putText(im, label, (6, 20), 0, 0.55, (255, 255, 255), 2); return im
T = max(len(rows[n]["blocking"]) for n in rows)
vw = cv2.VideoWriter(f"{OUT}/motion2.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 24, (1344, 384))
for _ in range(3):
    for t in range(T):
        vw.write(np.vstack([np.hstack([rd(v[t % len(v)], f"{n} {k}") for k, v in rows[n].items()]) for n in rows]))
vw.release()
!cd {OUT} && zip -q -r motion2_result.zip m2 motion2.mp4 motion2_flicker.json && ls -la {OUT}


In [ ]:
# 5) Colab이면 결과 zip 내려받기(Kaggle은 Output 탭에서 받음)
if COLAB and OUT.startswith("/content"):
    from google.colab import files
    files.download(f"{OUT}/motion2_result.zip")
